In [ ]:
import os
import subprocess
import torch
import gc
import shutil
import time
import re
import glob

from google.colab import userdata

# 1. Lấy nội dung chuỗi JSON từ Secrets của Colab
gcp_json_string = userdata.get('GCP_SERVICE_ACCOUNT')

# 2. Lưu nó ra một file tạm ẩn trên Colab
key_path = '/tmp/gcp_sa_key.json'
with open(key_path, 'w', encoding='utf-8') as f:
    f.write(gcp_json_string)

# 3. Trỏ biến môi trường đến file này
os.environ['GOOGLE_APPLICATION_CREDENTIALS'] = key_path

# Chạy đoạn này để GPU luôn hoạt động nhẹ, tránh bị Colab báo idle
import threading
def keep_gpu_busy():
    if not torch.cuda.is_available(): # Kiểm tra xem GPU có sẵn sàng không
        print("⚠️ Không tìm thấy GPU. Bỏ qua tác vụ giữ GPU hoạt động.")
        return
    print("🔋 Bắt đầu giữ GPU hoạt động ngầm...")

    a = torch.ones(1000, 1000).cuda() # Tạo một tensor nhỏ trên GPU
    while True:
        a = a @ a  # Thực hiện phép nhân ma trận đơn giản liên tục
        time.sleep(120) # Nghỉ 120s (2 phút) rồi làm tiếp để không tốn quá nhiều tài nguyên
t = threading.Thread(target=keep_gpu_busy) # Chạy trong luồng riêng biệt (background thread)
t.daemon = True
t.start()

In [ ]:
from google.colab import drive

# --- CẤU HÌNH ---
FOLDER_NAME = "Survival"
DRIVE_ROOT_PATH = "/content/drive/MyDrive/CharenjiZukan/2603"

# Đường dẫn nguồn (Drive) và đích (Local)
SOURCE_PARENT_PATH = os.path.join(DRIVE_ROOT_PATH)
LOCAL_PARENT_PATH = os.path.join("/content", FOLDER_NAME)
# 1. MOUNT DRIVE
drive.mount('/content/drive', force_remount=True)
# 2. LÀM SẠCH THƯ MỤC LOCAL CŨ
# Dùng subprocess rm -rf để xóa sạch thư mục cũ nếu có
subprocess.run(["rm", "-rf", LOCAL_PARENT_PATH], check=False)
subprocess.run(["rm", "-rf", "/content/assets"], check=False)
# Tạo lại thư mục cha rỗng ở local
os.makedirs(LOCAL_PARENT_PATH, exist_ok=True)

# Copy thư mục assets từ Drive về Local
print("📥 [COPY] Đang tải thư mục assets...")
subprocess.run(["cp", "-r", "/content/drive/MyDrive/CharenjiZukan/0 - assets", "/content/assets"], check=False)
print("✅ Hoàn tất copy thư mục assets!")

# 3. DUYỆT VÀ COPY THEO ĐIỀU KIỆN
if os.path.exists(SOURCE_PARENT_PATH):
    print(f"🚀 Bắt đầu đồng bộ từ: {SOURCE_PARENT_PATH}")
    # Lấy danh sách các thư mục con (Chapter)
    subfolders = sorted([f for f in os.listdir(SOURCE_PARENT_PATH)
                         if os.path.isdir(os.path.join(SOURCE_PARENT_PATH, f))])

    for item_name in subfolders:
        # Định nghĩa các đường dẫn cần thiết cho Item hiện tại
        source_item_path = os.path.join(SOURCE_PARENT_PATH, item_name)
        local_item_path = os.path.join(LOCAL_PARENT_PATH, item_name)
        # File/Folder cần kiểm tra trên Drive
        drive_output_folder = os.path.join(source_item_path, "output_sync")
        # --- KIỂM TRA ĐIỀU KIỆN ---
        # Kiểm tra xem trên Drive đã có thư mục output_sync chưa
        if os.path.exists(drive_output_folder) and os.path.isdir(drive_output_folder):
            print(f"⏩ [BỎ QUA] Đã có folder 'output_sync': {item_name}")
            continue # Nhảy sang vòng lặp kế tiếp
        else:
            print(f"📥 [COPY] Đang tải dữ liệu nguồn: {item_name}")
            # 1. Tạo thư mục đích tại Local (tương đương lệnh: mkdir -p "path")
            subprocess.run(["mkdir", "-p", local_item_path], check=True)
            # 2. Copy toàn bộ nội dung (files và folders) từ Drive về Local
            # Chúng ta sử dụng đường dẫn nguồn kết hợp với "/." để copy mọi thứ bên trong
            source_content = os.path.join(source_item_path, ".")
            try:
                # Lệnh cp -r sẽ copy đệ quy toàn bộ nội dung
                subprocess.run(["cp", "-r", source_content, local_item_path], check=True)
                print(f"  ✅ Đã copy toàn bộ nội dung của: {item_name}")
            except subprocess.CalledProcessError as e:
                print(f"  ❌ Lỗi khi copy tại {item_name}: {e}")
    print("\n✅ Hoàn tất quá trình copy!")
else:
    print(f"❌ Không tìm thấy thư mục gốc trên Drive: {SOURCE_PARENT_PATH}")

In [ ]:
!apt-get -y install fonts-noto-cjk
!fc-cache -fv

# Cài đặt uv
!curl -LsSf https://astral.sh/uv/install.sh | sh
import os
os.environ['PATH'] += ':/root/.local/bin'

# Clone project (Private Repository với Secrets)
from google.colab import userdata
token = userdata.get('github_token')
!git clone https://{token}@github.com/ThanhVoKim/CharenjiZukan.git /content/CharenjiZukan

!pip install -q pyyaml pytest

In [ ]:
%cd /content/CharenjiZukan
!uv pip install -e .

# Cài đặt rubberband-cli (cần cho time-stretch)
!apt-get install -y rubberband-cli

# Tạo virtual environment và cài whisperx
!uv venv
# !uv pip install -p .venv/bin/python whisperx
!uv pip install -p .venv/bin/python pyrubberband

# Cài CUDA dependencies
# !apt install libcudnn8 libcudnn8-dev -y

# Set environment variables
# %env MPLBACKEND=agg
# %env TORCH_FORCE_NO_WEIGHTS_ONLY_LOAD=true
# %env LD_LIBRARY_PATH=/usr/lib64-nvidia:/usr/local/lib/python3.12/dist-packages/nvidia/cudnn/lib/

# !uv run whisperx test.mp3 --language en --hf_token {hf_token} --diarize  --model large-v2 --align_model WAV2VEC2_ASR_LARGE_LV60K_960H

In [ ]:
#### 1a. Mute Audio

!uv run mute-srt \
    --input       /content/Survival/3/7509430509194890537.mp4 \
    --mute        /content/Survival/3/mute.srt \
    --output      /content/audio_muted.wav

In [ ]:
#### 1b. Extract Audio

!uv run extract-srt \
  --input /content/Survival/3/7509430509194890537.mp4 \
  --mute /content/Survival/3/mute.srt \
  --output /content/audio_extracted.wav

In [ ]:
#### Demucs Audio Separation (demucs-audio)

!uv run demucs-audio \
    --input   /content/Survival/1/7591460023444753707.mp4 \
    --output  /content/audio_vocals.wav \
    --model   htdemucs \
    --keep    vocals \
    --bitrate 192k \
    --device  cuda \
    --segment 7 \
    --verbose



In [ ]:
#### 4a. Translate Note

!uv run translate-srt \
    --input  /content/Survival/1/note.srt \
    --output /content/note_ja.srt \
    --provider  vertexai \
    --provider-config /content/CharenjiZukan/config/vertexai_translate.yaml \
    --lang      "Japanese" \
    --model     "gemini-3.1-pro-preview" \
    --batch     20 \
    --wait      3 \
    --max-chars 0 \
    --verbose

In [ ]:
#### 4b. Convert SRT to ASS

!uv run srt-to-ass \
    --input     /content/note_ja.srt \
    --output    /content/note_overlay.ass \
    --template  /content/CharenjiZukan/assets/sample.ass \
    --max-chars 14

In [ ]:
# @title
### Bước 5: Translate Subtitle
from google.colab import userdata
gemini_key = userdata.get('gemini_token')

!uv run translate-srt \
    --input     /content/Survival/1/subtitle.srt \
    --output    /content/subtitle_ja.srt \
    --provider  vertexai \
    --provider-config /content/CharenjiZukan/config/vertexai_translate.yaml \
    --lang      "Japanese" \
    --model     "gemini-3.1-pro-preview" \
    --batch     50 \
    --wait      3 \
    --max-chars 27 \
    --verbose


In [ ]:
### Bước 8: TTS

!uv run tts-srt --input /content/subtitle_translated_slow.srt --output /content/audioTTS_slow.wav --voice ja-JP-KeitaNeural --autorate --verbose

In [ ]:
!python -c "import google.genai, yaml, tenacity; print('OK: deps ready')"
!python -m pytest tests/test_translation_providers.py --collect-only -q -k "vertexai"
!python -m pytest tests/test_translation_providers.py -k "vertexai" -vv -rs

In [ ]:
from google.colab import userdata

# Cấp API key vào môi trường
os.environ["GEMINI_API_KEY"] = userdata.get('gemini_token')
os.environ["OPENAI_API_KEY"] = userdata.get('ramcloud_token')

# !python run_colab_tests.py --tags sync_engine
!python -m pytest tests/test_audio_assembler.py -v
# !python -m pytest tests/test_translation_providers.py::TestLayer4_RealAPIs::test_vertexai_real_api
# !uv run pytest tests/test_translation_providers.py -k "vertexai" -v

In [ ]:
!uv run sync-video \
    --video /content/sample-extract.mp4 \
    --subtitle /content/subtitle_translated.srt \
    --black-bg /content/CharenjiZukan/assets/black-background.jpg \
    --note-overlay-png /content/CharenjiZukan/assets/note-overlay.png \
    --note-overlay-ass /content/sample.ass \
    --mute /content/mute.srt \
    --ambient "/content/Lonely-Dance.m4a" \
    --output-dir /content/output_sync \
    --tts-voice ja-JP-KeitaNeural \
    --workers 4 \
    --keep-tmp \
    --subtitle-fontname "Noto Sans CJK JP"

In [ ]:
!uv pip install --upgrade transformers
!uv pip install qwen-vl-utils

In [ ]:
from google.colab import userdata
hf_token = userdata.get('hf_token')

# qwen-pixels ảnh thật: 196 (theo box OCR `subtitle 0 984 1920 80`)
!uv run video-ocr /content/Survival/1/7591460023444753707.mp4 \
    --output-dir /content/output \
    --config /content/CharenjiZukan/config/extractor_config.yaml \
    --boxes-file /content/CharenjiZukan/assets/boxesOCR.txt \
    --ocr-model Qwen/Qwen3-VL-8B-Instruct \
    --frame-interval 3 \
    --scene-threshold 1 \
    --phash_threshold 3 \
    --min-scene-frames 1 \
    --max-duration 15.0 \
    --batch-size 16 \
    --qwen-min-pixels 128 \
    --qwen-max-pixels 256 \
    --device cuda \
    --hf-token "{hf_token}" \
    --warn-english \
    --format srt \
    -vv
    # --cv-prefilter \


In [ ]:
!uv run sync-video \
    --video /content/Survival/1/7591460023444753707.mp4 \
    --subtitle /content/Survival/1/subtitle_ja.srt \
    --mute /content/Survival/1/mute.srt \
    --note-overlay-ass /content/Survival/1/note_overlay.ass \
    --note-overlay-png /content/assets/note_overlay.png \
    --black-bg /content/CharenjiZukan/assets/black-background.jpg \
    --ambient "/content/assets/Huma Huma Crimson Fly.mp3" \
    --output-dir /content/output_sync \
    --tts-voice ja-JP-KeitaNeural \
    --slow-cap 0.1 \
    --workers 4 \
    --keep-tmp \
    --subtitle-fontname "Noto Sans CJK JP"


In [ ]:
if os.path.exists('/content/output_sync/'): # Kiểm tra thư mục nguồn trước khi di chuyển
    !mv /content/output_sync/ /content/drive/MyDrive/CharenjiZukan/2603/1/
    print("✅ Đã di chuyển thư mục thành công!")

In [ ]:
SOURCE_DIR = "/content/output_sync"
DESTINATION_DIR = "/content/drive/MyDrive/CharenjiZukan/2603/1"

import os
import time
from google.colab import runtime
from google.colab import drive

if not os.path.exists(SOURCE_DIR):
    print(f"❌ LỖI: Không tìm thấy thư mục nguồn {SOURCE_DIR}")
else:
    # Đảm bảo thư mục đích tồn tại
    os.makedirs(DESTINATION_DIR, exist_ok=True)
    print(f"🚀 Bắt đầu copy siêu tốc từ {SOURCE_DIR} đến {DESTINATION_DIR}...")
    # -a: archive mode (giữ nguyên quyền, cấu trúc)
    # -P: hiện progress bar cho từng file
    # Lưu ý: Thêm dấu '/' vào cuối SOURCE_DIR để copy "nội dung bên trong"
    # chứ không copy cả cái vỏ thư mục 'output_sync'
    !rsync -avP "{SOURCE_DIR}/" "{DESTINATION_DIR}/"
    # Kiểm tra mã lỗi trả về của lệnh linux trước đó
    if _exit_code == 0:
        print("\n✅ Đã copy xong bằng rsync!")
        drive.flush_and_unmount()
        print("✅ Đã đồng bộ hoàn toàn lên Cloud và unmount Drive an toàn!")
        # NGẮT KẾT NỐI COLAB
        time.sleep(10)
        runtime.unassign()
    else:
        print(f"\n❌ Lệnh rsync thất bại với mã lỗi: {_exit_code}")
